# **Question 5: Orchestration & Zero-Downtime Deployments**

**Focus:** **Signals, Graceful Shutdowns, and Healthchecks**

**Scenario:**
You are deploying an update to a high-traffic Node.js payment service using Docker Swarm (or Kubernetes, the concept is identical). You trigger a **Rolling Update** to replace `v1` containers with `v2`.

**The Issue:**
During the update, monitoring shows a spike in **502 Bad Gateway** errors and **failed transactions** for about 5-10 seconds.
It turns out that when Docker stops the old `v1` container, it kills the process immediately, severing active payment connections.

**Question:**
1.  **The Signal Flow:** When you issue a command to stop a container (or when an orchestrator rotates it), what is the **first Linux Signal** Docker sends to PID 1? What happens if the app ignores it? What is the **second Signal** Docker sends, and after how long (default)?
2.  **The Fix:** How do you modify your application code (Node.js/Python/Go) to handle this signal correctly to ensure **Zero Downtime**? (This concept is called "Graceful Shutdown").
3.  **The Docker Config:** If your application takes 45 seconds to finish processing a transaction, but Docker's default timeout is shorter, how do you adjust this in the `Dockerfile` or `docker run` command?

---

## Background (Why This Matters in MLOps / Backend)

During a rolling update of a high-traffic payment service:
- Old container is stopped
- Active payment connections are cut mid-processing
- Users see **502 Bad Gateway**
- Transactions fail

This is **not** a deployment problem.
This is a **process lifecycle & signal handling** problem.

Understanding signal flow is what separates a junior who "just deploys" from a senior who designs zero-downtime systems.

---

## Q1 — The Signal Flow (What Actually Happens)

### Step-by-Step

When you run `docker stop my-container` — or when Kubernetes/Docker Swarm rotates a pod — the following happens:

```
Step 1:  Docker sends  →  SIGTERM  →  to PID 1 inside the container
Step 2:  Docker waits  →  10 seconds (default grace period)
Step 3:  If app ignores SIGTERM  →  Docker sends SIGKILL
```

### SIGTERM — The Polite Request

| Property | Detail |
|---|---|
| Signal | `SIGTERM` (signal 15) |
| Meaning | "Please shut down cleanly" |
| Can be caught? | ✅ Yes — your app can listen and handle it |
| Can be ignored? | ✅ Yes — but that's the bug |
| Purpose | Gives app time to finish in-flight requests, close DB connections, flush logs |

### SIGKILL — The Force Kill

| Property | Detail |
|---|---|
| Signal | `SIGKILL` (signal 9) |
| Meaning | "Die now, no exceptions" |
| Can be caught? | ❌ No — kernel enforces it directly |
| Can be ignored? | ❌ No |
| Purpose | Guarantees the container eventually stops — prevents zombie containers |
| Sent after | **10 seconds** of SIGTERM being ignored (Docker default) |

### Why You Saw 502 Errors

```
Payment service was serving live traffic
        ↓
Received SIGTERM
        ↓
App ignored it (no signal handler)
        ↓
Orchestrator removed container from load balancer
        ↓
Active connections were severed
        ↓
After 10s → SIGKILL
        ↓
In-flight transactions killed mid-processing
        ↓
502 Bad Gateway + Failed transactions
```

### Interview Phrasing

> "Docker sends `SIGTERM` first — a polite termination request that the app can catch and handle. If the app ignores it, Docker waits 10 seconds (the default grace period) and then sends `SIGKILL`, which cannot be caught or ignored and forcefully terminates the process. The 502s happen because connections are severed before in-flight requests complete."

---

## Q2 — The Fix: Graceful Shutdown in Code

**Concept:** When your app receives `SIGTERM`, it should:
1. **Stop accepting new connections**
2. **Finish all active in-flight requests**
3. **Close DB connections / flush logs**
4. **Exit cleanly with code 0**

### Node.js

```javascript
const express = require('express');
const app = express();

const server = app.listen(3000, () => {
  console.log('Server running on port 3000');
});

process.on('SIGTERM', () => {
  console.log('SIGTERM received. Shutting down gracefully...');

  // Step 1: Stop accepting new connections
  server.close(() => {
    // Step 2: All active connections finished
    console.log('All connections closed. Process terminating.');
    process.exit(0);
  });

  // Optional: force exit if connections hang too long
  setTimeout(() => {
    console.error('Forcing exit after timeout');
    process.exit(1);
  }, 30000);
});
```

### Python (FastAPI / Flask)

```python
import signal
import sys
import uvicorn
from fastapi import FastAPI

app = FastAPI()

# Store server reference for graceful shutdown
server = None

def handle_sigterm(signum, frame):
    print("SIGTERM received. Shutting down gracefully...")
    # Flush logs, close DB connections here
    if server:
        server.should_exit = True
    sys.exit(0)

# Register the signal handler
signal.signal(signal.SIGTERM, handle_sigterm)
signal.signal(signal.SIGINT, handle_sigterm)   # also handle Ctrl+C

if __name__ == "__main__":
    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000))
    server.run()
```

### Go

```go
package main

import (
    "context"
    "fmt"
    "net/http"
    "os"
    "os/signal"
    "syscall"
    "time"
)

func main() {
    srv := &http.Server{Addr: ":8080"}

    // Run server in goroutine
    go func() {
        if err := srv.ListenAndServe(); err != http.ErrServerClosed {
            panic(err)
        }
    }()

    // Block until SIGTERM or SIGINT
    ctx, stop := signal.NotifyContext(context.Background(), os.Interrupt, syscall.SIGTERM)
    defer stop()
    <-ctx.Done()

    fmt.Println("SIGTERM received. Shutting down gracefully...")

    // Give active requests 30s to complete
    shutdownCtx, cancel := context.WithTimeout(context.Background(), 30*time.Second)
    defer cancel()

    if err := srv.Shutdown(shutdownCtx); err != nil {
        fmt.Printf("Forced shutdown: %v\n", err)
    }

    fmt.Println("Server stopped cleanly.")
}
```

---

## The Critical PID 1 Trap (Senior-Level Detail)

### Why PID 1 Matters

Inside a container, your app runs as **PID 1**. PID 1 has special behavior in Linux — it does **not** automatically forward signals to child processes the way a normal process manager would.

### The Shell Form Bug

```dockerfile
# ❌ WRONG — Shell form
CMD node app.js
# This becomes: /bin/sh -c "node app.js"
# sh becomes PID 1
# sh does NOT forward SIGTERM to node
# Your app never receives the signal → SIGKILL after 10s
```

```dockerfile
# ✅ CORRECT — Exec form (JSON array)
CMD ["node", "app.js"]
# node becomes PID 1 directly
# SIGTERM is received by your app
```

### The Rule
> Always use **exec form** (`CMD ["executable", "arg"]`) in Dockerfiles, never **shell form** (`CMD executable arg`).

### The Init Pattern (Staff/Principal Level)

If your app genuinely cannot handle PID 1 responsibilities (zombie process reaping, signal forwarding) — use a lightweight init system:

```dockerfile
# Option 1: tini (explicit)
FROM node:18
RUN apt-get install -y tini
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["node", "app.js"]

# Option 2: Docker built-in init flag
# docker run --init my-image

# Option 3: dumb-init
ENTRYPOINT ["dumb-init", "--"]
CMD ["node", "app.js"]
```

> `tini` and `dumb-init` are minimal init systems that correctly forward signals and reap zombie processes — solving the PID 1 problem without changing your application code.

---

## Q3 — Adjusting the Timeout (45-Second Transactions)

Docker's default grace period is **10 seconds**. If your payment processing needs 45 seconds, Docker kills it too early.

### Solution 1 — `docker run` flag

```bash
docker run --stop-timeout=60 my-payment-service
```

### Solution 2 — Dockerfile (`STOPSIGNAL`)

```dockerfile
# Sets which signal Docker sends (not the timeout)
# Useful if your app uses a different signal
STOPSIGNAL SIGTERM
```

> Note: `STOPSIGNAL` changes the *signal type*, not the timeout. Use it when your app listens on a custom signal.

### Solution 3 — Docker Compose

```yaml
services:
  payment-service:
    image: my-payment-service:v2
    stop_grace_period: 60s    # wait 60s before SIGKILL
```

### Solution 4 — Kubernetes

```yaml
spec:
  terminationGracePeriodSeconds: 60   # default is 30
  containers:
    - name: payment-service
      image: my-payment-service:v2
```

### Summary Table

| Environment | Config | Default |
|---|---|---|
| `docker run` | `--stop-timeout=60` | 10s |
| Docker Compose | `stop_grace_period: 60s` | 10s |
| Kubernetes | `terminationGracePeriodSeconds: 60` | 30s |
| Dockerfile | `STOPSIGNAL SIGTERM` | signal type only |

---

## Bonus: Healthchecks (Senior Concept)

Healthchecks tell the orchestrator whether a container is ready to receive traffic. Without them, traffic is routed to a container that hasn't finished starting up — causing 502s on the *other* end of the update.

```dockerfile
HEALTHCHECK --interval=10s --timeout=3s --retries=3 \
  CMD curl -f http://localhost:3000/health || exit 1
```

```python
# FastAPI health endpoint
@app.get("/health")
def health():
    return {"status": "ok"}
```

```yaml
# Kubernetes readiness probe (more granular than HEALTHCHECK)
readinessProbe:
  httpGet:
    path: /health
    port: 8000
  initialDelaySeconds: 5
  periodSeconds: 10
```

> **Liveness probe** → "Is the container alive?" (restart if failing)
> **Readiness probe** → "Is the container ready for traffic?" (remove from LB if failing)

---

## Full Zero-Downtime Rolling Update Flow

```
1. v2 container starts
        ↓
2. Healthcheck passes on v2
        ↓
3. Load balancer shifts traffic to v2
        ↓
4. v1 receives SIGTERM
        ↓
5. v1 stops accepting NEW connections
        ↓
6. v1 finishes all ACTIVE connections (within grace period)
        ↓
7. v1 exits cleanly (process.exit(0))
        ↓
8. No dropped connections. No 502s. No failed transactions.
```

---

## Senior Engineer Answer (Full Interview Script)

> "When Docker stops a container — either via `docker stop` or an orchestrator during a rolling update — it first sends `SIGTERM` to PID 1 inside the container. This is a polite termination signal that the application can catch. If the app ignores it, Docker waits 10 seconds by default and then sends `SIGKILL`, which cannot be caught or ignored and immediately terminates the process.
>
> The 502s happen because the app had no `SIGTERM` handler — when the container was removed from the load balancer, active connections were severed mid-request.
>
> The fix is graceful shutdown: listen for `SIGTERM`, stop accepting new connections, let active ones finish, then exit. In Node.js this is `server.close()` inside a `process.on('SIGTERM')` handler.
>
> There's also a critical PID 1 trap: if you use shell form in your Dockerfile (`CMD node app.js`), the shell becomes PID 1 and doesn't forward `SIGTERM` to your app. You must use exec form: `CMD ["node", "app.js"]`.
>
> If the app needs more than 10 seconds — say 45 seconds for a payment transaction — you configure `stop_grace_period: 60s` in Docker Compose or `terminationGracePeriodSeconds: 60` in Kubernetes.
>
> And for apps that can't handle PID 1 responsibilities at all, you use a lightweight init system like `tini` or Docker's built-in `--init` flag."

---

## One-Line Revision Summary

> Docker sends `SIGTERM` → app should stop accepting connections and drain active ones → if ignored, `SIGKILL` fires after 10 seconds → fix by registering a `SIGTERM` handler (graceful shutdown) → use exec form in Dockerfile so PID 1 receives the signal → extend grace period via `stop_grace_period` or `terminationGracePeriodSeconds` for long-running transactions.

---

## Interview Delivery Tips

1. **Lead with the signal flow:** "SIGTERM first, 10 seconds, then SIGKILL" — say this confidently and immediately.
2. **The PID 1 / exec form trap** is the #1 differentiator — most candidates miss it entirely. Mention shell form vs exec form explicitly.
3. **Name the init systems:** `tini`, `dumb-init`, `docker run --init` — these signal real production experience.
4. **Readiness vs Liveness probes** in Kubernetes — shows you understand the orchestrator's traffic management, not just signal handling.
5. **Connect to transactions:** "In a payment service, a forced SIGKILL mid-transaction can cause double charges or partial writes — graceful shutdown is a correctness requirement, not just a UX nicety."